# Lecture 2: Data Wrangling, Git, and Tabular Data


## Setup: imports and California Housing dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)

housing = fetch_california_housing(as_frame=True)
X = housing.data.copy()
y = housing.target.copy()

df = X.copy()
df["MedHouseVal"] = y
df.head()

## 1. Version control with Git

Git helps you track changes, collaborate, and keep a reliable history of your work.

```{important}
Git is great for **code and text**, but it is not designed for large binary artifacts (big datasets, model checkpoints).
```

### Minimal Git commands

```bash
git init
git status
git add <files>
git commit -m "message"
git log --oneline --decorate --graph
git checkout -b <new-branch>
git pull
git push
```

### Merge conflicts

Merge conflicts happen when Git cannot automatically combine changes.
Resolve by editing the file(s), then:

```bash
git add <resolved-file>
git commit
```

```{warning}
After resolving conflicts in notebooks, **re-run** the notebook to ensure it still executes top-to-bottom.
```

### `.gitignore` and large files

Use `.gitignore` to exclude build artifacts, caches, credentials, and large data:

```text
__pycache__/
.ipynb_checkpoints/
*.pyc
data/
models/
.env
```

For large binaries, consider:
- external storage (object store, dataset registry)
- **Git LFS** (only if your platform and team support it)

```{tip}
Agree early in a project what belongs in Git and what doesn't.
```

## 2. Data wrangling: definition and motivation

Data wrangling turns raw data into analysis-ready data by cleaning, transforming, reshaping, and combining sources.

```{note}
In many projects, data wrangling takes most of the time—even more than modelling.
```

## 3. Tidy data (wide vs long)

Tidy data principles:
1. Variables → columns  
2. Observations → rows  
3. One observational unit → one table  

We'll demonstrate wide→long reshaping using the California Housing dataset.

In [ ]:
df2 = df.copy()
df2["MedInc_bin"] = pd.cut(
    df2["MedInc"],
    bins=[0, 2, 4, 6, 8, 15],
    labels=["0-2", "2-4", "4-6", "6-8", "8-15"],
    include_lowest=True,
)

wide = (
    df2.groupby("MedInc_bin", observed=True)
       .agg(
           mean_value=("MedHouseVal", "mean"),
           median_value=("MedHouseVal", "median"),
           mean_rooms=("AveRooms", "mean"),
           n=("MedHouseVal", "size"),
       )
       .reset_index()
)
wide

In [ ]:
long = wide.melt(
    id_vars=["MedInc_bin", "n"],
    var_name="metric",
    value_name="value",
)
long.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for metric, sub in long.groupby("metric"):
    ax.plot(sub["MedInc_bin"].astype(str), sub["value"], marker="o", label=metric)

ax.set_xlabel("MedInc_bin")
ax.set_title("Wide → long (tidy) example on California Housing")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Core DataFrame operations

Typical data-wrangling building blocks:
- selecting columns
- filtering rows
- creating new columns
- sorting
- grouping & aggregating

In [ ]:
df_sel = df[["MedInc", "HouseAge", "AveRooms", "AveOccup", "MedHouseVal"]].copy()
df_filt = df_sel[df_sel["MedInc"].between(3, 6)].copy()
df_filt["rooms_per_person"] = df_filt["AveRooms"] / df_filt["AveOccup"]
df_filt.head()

### Grouping and aggregation

In [ ]:
summary = (
    df2.groupby("MedInc_bin", observed=True)
       .agg(
           mean_value=("MedHouseVal", "mean"),
           std_value=("MedHouseVal", "std"),
           mean_age=("HouseAge", "mean"),
           mean_occupancy=("AveOccup", "mean"),
       )
       .reset_index()
)
summary

## 5. Missing values (nulls)

Missing values can appear due to measurement failures, parsing issues, joins, or masked data.
We'll inject missing values and show common handling patterns.

In [ ]:
df_missing = df.copy()
rng = np.random.default_rng(0)
mask = rng.random(len(df_missing)) < 0.03  # 3% missing
df_missing.loc[mask, "AveRooms"] = np.nan

df_missing.isna().mean().sort_values(ascending=False).head(10)

In [ ]:
dropped = df_missing.dropna(subset=["AveRooms"])
imputed = df_missing.copy()
imputed["AveRooms"] = imputed["AveRooms"].fillna(imputed["AveRooms"].median())

print("AveRooms missing (original):", df_missing["AveRooms"].isna().sum())
print("Rows after drop:", len(dropped))
print("AveRooms missing (imputed):", imputed["AveRooms"].isna().sum())

## 6. Strings, regex, and dates

California Housing is numeric-only, so we'll create derived string/date fields to demonstrate:
- string formatting
- regex extraction
- datetime parsing and components

In [ ]:
df_str = df.reset_index(drop=False).rename(columns={"index": "row_id"}).copy()
df_str["tract_code"] = df_str["row_id"].map(lambda i: f"TRACT-{i:06d}-CA")
df_str["collection_date"] = pd.Timestamp("2024-01-01") + pd.to_timedelta(df_str["row_id"] % 180, unit="D")

df_str["tract_id"] = df_str["tract_code"].str.extract(r"TRACT-(\d+)-CA")[0].astype(int)
df_str["weekday"] = df_str["collection_date"].dt.day_name()

df_str[["tract_code", "tract_id", "collection_date", "weekday"]].head()

```{hint}
Regex extraction is useful when keys encode structure (region codes, timestamps, product IDs, …).
```

## 7. Joining / merging tables

We'll create a lookup table and merge it onto grouped results.

In [ ]:
bin_lookup = pd.DataFrame({
    "MedInc_bin": ["0-2", "2-4", "4-6", "6-8", "8-15"],
    "bin_label": ["low", "lower-mid", "mid", "upper-mid", "high"],
})

summary2 = summary.merge(bin_lookup, on="MedInc_bin", how="left")
assert summary2["bin_label"].isna().sum() == 0
summary2

## 8. SQL for tabular data (runnable with sqlite3)

Many wrangling operations can be expressed in SQL. We'll run a simple GROUP BY query via SQLite.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
df_sql = df[["MedInc", "AveRooms", "MedHouseVal"]].copy()
df_sql.to_sql("housing", conn, index=False, if_exists="replace")

query = '''
SELECT
  CASE
    WHEN MedInc < 2 THEN '0-2'
    WHEN MedInc < 4 THEN '2-4'
    WHEN MedInc < 6 THEN '4-6'
    WHEN MedInc < 8 THEN '6-8'
    ELSE '8-15'
  END AS MedInc_bin,
  COUNT(*) AS n,
  AVG(MedHouseVal) AS mean_value,
  AVG(AveRooms) AS mean_rooms
FROM housing
GROUP BY MedInc_bin
ORDER BY MedInc_bin
'''
pd.read_sql_query(query, conn)

## 9. Polars (optional)

This cell runs even if Polars is not installed.

In [ ]:
try:
    import polars as pl

    pl_df = pl.from_pandas(df[["MedInc", "AveRooms", "MedHouseVal"]])

    out = (
        pl_df
        .with_columns([
            pl.when(pl.col("MedInc") < 2).then(pl.lit("0-2"))
              .when(pl.col("MedInc") < 4).then(pl.lit("2-4"))
              .when(pl.col("MedInc") < 6).then(pl.lit("4-6"))
              .when(pl.col("MedInc") < 8).then(pl.lit("6-8"))
              .otherwise(pl.lit("8-15"))
              .alias("MedInc_bin")
        ])
        .group_by("MedInc_bin")
        .agg([
            pl.len().alias("n"),
            pl.mean("MedHouseVal").alias("mean_value"),
            pl.mean("AveRooms").alias("mean_rooms"),
        ])
        .sort("MedInc_bin")
    )
    display(out)
except Exception as e:
    print("Polars not available; skipping optional demo.")
    print("Install with: pip install polars")

## 10. Data formats: CSV vs Parquet

Parquet is often faster and smaller than CSV for analytics. We'll attempt a Parquet round-trip.

In [ ]:
import tempfile

tmpdir = tempfile.TemporaryDirectory()
parquet_path = os.path.join(tmpdir.name, "housing_sample.parquet")

sample = df.head(5000)

try:
    sample.to_parquet(parquet_path, index=False)
    restored = pd.read_parquet(parquet_path)
    print("Parquet round-trip OK:", restored.shape)
    restored.head()
except Exception as e:
    print("Parquet not available here. Install `pyarrow` (recommended).")
    print("Error:", repr(e))